In [ ]:
from medlat import available_models, get_model, MODEL_REGISTRY



import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import make_grid
from torchvision.datasets import STL10
from medmnist import PathMNIST
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import torch.nn.functional as F
from medmnist import INFO

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"


### Prep Dataset ###
IMG_SIZE = 96
tensor_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE,IMG_SIZE)),
        transforms.ToTensor(),
    ]
)

train_set = STL10(root=".", split="train", transform=tensor_transforms, download=True)
test_set = STL10(root=".", split="test", transform=tensor_transforms, download=True)

# Take small subsets (100 samples each) for quick prototyping
from torch.utils.data import Subset

train_subset = Subset(train_set, np.random.choice(len(train_set), 100, replace=False))
test_subset = Subset(test_set, np.random.choice(len(test_set), 100, replace=False))

train_set = train_subset
test_set = train_subset

### Set Device ###
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def train_ae(model,
          train_set,
          test_set,
          batch_size,
          num_epochs,
          evaluation_iterations,
          step_break=100_000):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)

    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=8)

    optimizer = optim.Adam(model.parameters(), lr=0.00001)

    train_losses = []
    evaluation_losses = []

    steps_per_epoch = len(trainloader)
    total_steps = steps_per_epoch * num_epochs
    step_counter = 0
    train_loss_running = []
    evaluation_loss_running = []

    with tqdm(total=num_epochs, desc="Epochs") as epoch_bar:
        for epoch in range(num_epochs):
            model.train()
            with tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{num_epochs} | loss: N/A", leave=False) as pbar:
                for images, labels in trainloader:
                    # print(labels)
                    images = images.to(device)

                    recon, extra_loss = model(images)
                    loss = F.mse_loss(recon, images) +  extra_loss
                    # Update tqdm description with current loss value
                    pbar.set_description(f"Epoch {epoch+1}/{num_epochs} | loss: {loss.item():.4f}")

                    train_loss_running.append(loss.item())

                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()

                    step_counter += 1
                    pbar.update(1)

                    if step_counter % step_break == 0:
                        break

                    # Evaluate every evaluation_iterations global step
                    if step_counter % evaluation_iterations == 0:
                        model.eval()
                        evaluation_loss_running.clear()
                        example_shown = False
                        with torch.no_grad():
                            for images_eval, labels_eval in testloader:
                                images_eval = images_eval.to(device)
                                recon_eval, _ = model(images_eval)
                                # Compute PSNR loss
                                mse = F.mse_loss(recon_eval, images_eval)
                                # Clamp to avoid log(0); add small epsilon
                                epsilon = 1e-8
                                psnr = 10 * torch.log10(1.0 / (mse + epsilon))
                                loss_eval = psnr
                                evaluation_loss_running.append(loss_eval.item())
                                
                                if not example_shown:
                                    n_show = min(16, images_eval.shape[0])
                                    pairs = torch.stack([images_eval[:n_show], recon_eval[:n_show]], dim=1)
                                    grid_tensor = pairs.reshape(n_show * 2, *pairs.shape[2:]).cpu()
                                    grid_img = make_grid(grid_tensor, nrow=8, padding=2, normalize=True)
                                    arr = grid_img.permute(1, 2, 0).numpy()
                                    plt.imshow(arr.squeeze() if arr.shape[-1] == 1 else arr, cmap="gray" if arr.shape[-1] == 1 else None)
                                    plt.axis("off")
                                    plt.title(f"Eval Batch at Step {step_counter}")
                                    plt.show()

                                    example_shown = True
                        
                                break
                        model.train()
                        
            if step_counter % step_break == 0:
                break
            epoch_bar.update(1)
            



    return model, train_losses, evaluation_losses

In [ ]:


all_models = available_models()
print(all_models)
f4_models = [name for name in all_models if "f4" in name]
print("All models with 'f4' in the name:", f4_models)

model_list=[
"continuous.aekl.f4_d3", 
# "continuous.aekl.f8_d4", "continuous.aekl.f16_d8",
# "continuous.aekl.f4_d8", "continuous.aekl.f4_d16", "continuous.aekl.f4_d32",
# "continuous.aekl.f8_d8", "continuous.aekl.f8_d16", "continuous.aekl.f8_d32",
# "continuous.aekl.f16_d16", "continuous.aekl.f16_d32", "continuous.aekl.f16_d64",
# "continuous.medvae.f8_d16", "continuous.medvae.f8_d32", "continuous.vavae.f8_d16_dinov2", "continuous.vavae.f8_d32_dinov2",
# "discrete.lfq.f4_d10_b10", "discrete.lfq.f8_d14_b14", "discrete.lfq.f16_d18_b18",
# "discrete.bsq.f4_d10_b10", "discrete.bsq.f8_d14_b14", "discrete.bsq.f16_d18_b18",
# "discrete.simple_qinco.f16_d8_e16384", "discrete.simple_qinco.f4_d3_e8192", "discrete.simple_qinco.f8_d4_e16384",
# "discrete.simvq.f16_d8_e16384", "discrete.simvq.f4_d3_e8192", "discrete.simvq.f8_d4_e16384",
# "discrete.vq.f16_d8_e16384", "discrete.vq.f4_d3_e8192", "discrete.vq.f8_d4_e16384",
# "token.detok.bs",
# "token.maetok.b_128_p8", # "token.maetok.b_256", "token.maetok.b_512",
# "token.titok.b_128_e2e", #"token.titok.b_256_e2e", "token.titok.b_512_e2e",
# "discrete.hcvq.residual_vq.S_16",
# "discrete.hcvq.soft_vq.S_16",
# "discrete.hcvq.grouped_vq.S_16",
# "continuous.wqvae.f8_d4_e16384",
# "continuous.soft_vq.f8_d16_e16384",
# "continuous.soft_vq.f8_d32_e16384",
# "continuous.soft_vq.f8_d16_e16384_dinov2",
# "continuous.soft_vq.f8_d32_e16384_dinov2",
# "continuous.soft_vq.f8_d16_e16384_biomedclip",
# "continuous.soft_vq.f8_d32_e16384_biomedclip",
# "discrete.hcvq.msrq.S_16",

# "token.vmae.t_p8_d16",
# "discrete.rqvae.f4_d3_e8192",
# "discrete.rqvae.f8_d4_e16384",
# "discrete.rqvae.f16_d8_e16384",
# "discrete.qinco.f4_d3_e8192",
# "discrete.qinco.f8_d4_e16384",
# "discrete.qinco.f16_d8_e16384",
# "continuous.dcae.f128c512",
# "token.softvq.b_t32_d32",
# "continuous.soft_vq.f4_d3_e8192",
# "discrete.rsimple_qinco.f16_d8_e16384", "discrete.rsimple_qinco.f4_d3_e8192", "discrete.rsimple_qinco.f8_d4_e16384",
]


print(model_list.__len__())
for model_name in model_list:
    print(f"Training: ", model_name)
    ae_model = get_model(model_name, img_size=(128, 128))
    # x = torch.randn(1, 3, 160, 176, 144)
    # recon, loss = f4(x)
    # print(recon.shape)
    # print(loss)
    linear_ae, train_losses, evaluation_losses = train_ae(ae_model,
                                                    train_set=train_set,
                                                    test_set=test_set,
                                                    batch_size=32,
                                                    num_epochs=100000,
                                                    evaluation_iterations=100,
                                                    step_break=5000)

In [ ]:
from medlat import create_scheduler, GenerativeScheduler

def train_generator(
    model,
    train_set,
    test_set,
    batch_size,
    num_epochs,
    evaluation_iterations,
    break_step,
    scheduler: GenerativeScheduler = None,   # pass in or falls back to flow matching
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)

    if scheduler is None:
        scheduler = create_scheduler(
            "flow",
            path_type="Linear",
            prediction="velocity",
            loss_weight=None,
            use_cosine_loss=True,
            use_lognorm=True,
        )

    trainloader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=8)
    testloader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=8)
    optimizer   = optim.Adam(model.parameters(), lr=0.0001)

    train_losses, evaluation_losses = [], []
    steps_per_epoch = len(trainloader)
    step_counter = 0
    train_loss_running, evaluation_loss_running = [], []

    with tqdm(total=num_epochs, desc="Epochs") as epoch_bar:
        for epoch in range(num_epochs):
            model.train()
            with tqdm(total=steps_per_epoch, desc=f"Epoch {epoch+1}/{num_epochs} | loss: N/A", leave=False) as pbar:
                for images, labels in trainloader:
                    images, labels = images.to(device), labels.to(device)
                    cond = dict(y=labels)

                    z    = model.vae_encode(images)
                    loss = scheduler.training_losses(model, z, model_kwargs=cond)["loss"].mean()

                    pbar.set_description(f"Epoch {epoch+1}/{num_epochs} | loss: {loss.item():.4f}")
                    train_loss_running.append(loss.item())

                    loss.backward()
                    optimizer.step()
                    optimizer.zero_grad()
                    step_counter += 1
                    pbar.update(1)

                    if step_counter % break_step == 0:
                        break

                    if step_counter % evaluation_iterations == 0:
                        model.eval()
                        evaluation_loss_running.clear()
                        with torch.no_grad():
                            sample_cond = dict(y=labels[:1].repeat(batch_size))
                            z_noise     = torch.randn(batch_size, *model.vae_encode(images[:1]).shape[1:], device=device)
                            samples     = scheduler.p_sample_loop(
                                model.generator.forward, z_noise.shape,
                                noise=z_noise, model_kwargs=sample_cond,
                                sampler="euler", num_steps=50,
                            )
                            out = model.vae_decode(samples)

                        import matplotlib.pyplot as plt
                        import math

                        batch_to_show = out.detach().cpu().numpy()
                        n_samples = batch_to_show.shape[0]
                        n_cols = min(8, n_samples)
                        n_rows = math.ceil(n_samples / n_cols)

                        fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
                        axes = np.array(axes).reshape(-1)  # flatten for easy indexing

                        for i in range(n_samples):
                            img = batch_to_show[i]
                            if img.ndim == 3 and img.shape[0] in [1, 3]:
                                img = np.transpose(img, (1, 2, 0))
                            axes[i].imshow(np.clip(img, 0, 1), cmap=None if img.ndim == 3 else "gray")
                            axes[i].axis("off")

                        # Hide unused subplots if any
                        for i in range(n_samples, n_rows * n_cols):
                            axes[i].axis("off")

                        plt.tight_layout()
                        plt.show()

                        train_losses.append(np.mean(train_loss_running) if train_loss_running else 0.0)
                        evaluation_losses.append(np.mean(evaluation_loss_running) if evaluation_loss_running else 0.0)
                        train_loss_running = []
                        model.train()

            if step_counter % break_step == 0:
                break
            epoch_bar.update(1)

    print("Final Training Loss",    train_losses[-1]      if train_losses      else "No log")
    print("Final Evaluation Loss",  evaluation_losses[-1] if evaluation_losses else "No log")
    return model, train_losses, evaluation_losses

In [ ]:
from medlat import GenWrapper


print(f"training generator")
generator = get_model("dit.b_2", learn_sigma=False, img_size=IMG_SIZE, vae_stride=ae_model.vae_stride, in_channels=ae_model.embed_dim)
wrapper = GenWrapper(generator, ae_model)

num_params = sum(p.numel() for p in generator.parameters())
print(f"Number of parameters in generator: {num_params}")

linear_ae, train_losses, evaluation_losses = train_generator(wrapper,
                                                        train_set=train_set,
                                                        test_set=test_set,
                                                        batch_size=32,
                                                        num_epochs=1000,
                                                        evaluation_iterations=100,
                                                        break_step=10_000)